In [27]:
year_month = "202401"

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 29, Finished, Available, Finished, False)

In [28]:
import requests
import zipfile
import os

import unicodedata
import re

import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date, regexp_replace, trim

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 30, Finished, Available, Finished, False)

In [46]:
url = f"https://portaldatransparencia.gov.br/download-de-dados/despesas-execucao/{year_month}"
bronze_destination_path = f"/lakehouse/default/Files/bronze/despesa_{year_month}"
temporary_zip_file = f"/tmp/despesas_{year_month}.zip"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

try:
    os.makedirs(bronze_destination_path, exist_ok=True)

    with requests.get(url, stream=True, headers=headers) as response:
        response.raise_for_status()

        with open(temporary_zip_file, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    print(f"Download do zip {year_month} concluído.")

    with zipfile.ZipFile(temporary_zip_file, "r") as zip_ref:
        zip_ref.extractall(bronze_destination_path)

    print(f"ZIP extraído com sucesso em: {bronze_destination_path}")

except Exception as e:
    print(f"Erro no processamento do mês {year_month}: {e}")

finally:
    if os.path.exists(temporary_zip_file):
        os.remove(temporary_zip_file)
        print("Arquivo temporário removido.")

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 48, Finished, Available, Finished, False)

Download do zip 202401 concluído.
ZIP extraído com sucesso em: /lakehouse/default/Files/bronze/despesa_202401
Arquivo temporário removido.


In [47]:
df_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "latin1")
    .load(f"Files/bronze/despesa_{year_month}/{year_month}_Despesas.csv")
)

display(df_bronze)

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1f9fa34-13fb-42df-82f2-3cbc6e225ac6)

In [31]:
def normalize_columns(dataframe):
    for df_bronze_col in dataframe.columns:
        nfkd_form = unicodedata.normalize('NFKD', df_bronze_col)
        text_no_accents = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
        
        text_lower = text_no_accents.lower()
        
        clean_text = re.sub(r'[^a-z0-9]', '_', text_lower)
        
        clean_text = re.sub(r'_+', '_', clean_text).strip('_')
        
        dataframe = dataframe.withColumnRenamed(
            df_bronze_col, 
            clean_text
        )
        
    return dataframe

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 33, Finished, Available, Finished, False)

In [32]:
df_bronze = normalize_columns(df_bronze)

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 34, Finished, Available, Finished, False)

In [33]:
df_bronze = (

    df_bronze

    .withColumn(
        "ano_mes",
        F.lit(year_month)
    )

    .withColumn(
        "ingestion_date",
        F.current_date()
    )

    .withColumn(
        "processing_timestamp",
        F.current_timestamp()
    )
    .withColumn(
    "source_file",
    F.input_file_name()
)
)

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 35, Finished, Available, Finished, False)

In [34]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano_mes")
    .option(
        "replaceWhere",
        f"ano_mes = {year_month}"
    )
    .saveAsTable("bronze_despesas")
)

StatementMeta(, f66f37af-04e5-4444-86c2-80848582007f, 36, Finished, Available, Finished, False)